In [ ]:
!pip install requests beautifulsoup4 -q

import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import os
import re
from datetime import datetime

print('✅ Libraries ready')

✅ Libraries ready


In [ ]:
import os
from google.colab import drive

# Clear the stuck mount
!fusermount -u /content/drive 2>/dev/null
!rm -rf /content/drive

drive.mount('/content/drive')

OUTPUT_FOLDER = '/content/drive/MyDrive/arc_of_conflict/data'
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/120.0.0.0 Safari/537.36'
}

# START_DATE = '2022-01-01'
# END_DATE   = '2025-12-31'

# START_DATE = '2010-01-01'
# END_DATE   = '2011-12-31'
# PHASE = 'P1'
# START_DATE = '2016-01-01'
# END_DATE   = '2019-12-31'
# PHASE = 'P3'
START_DATE = '2022-01-01'
END_DATE   = '2025-12-31'
PHASE = 'P4'

print('✅ Configuration set')

Mounted at /content/drive
✅ Configuration set


In [ ]:
#

import xml.etree.ElementTree as ET
from datetime import date, timedelta

def get_aljazeera_from_sitemaps():
    all_urls = []

    start = date.fromisoformat(START_DATE)
    end = date.fromisoformat(END_DATE)
    delta = timedelta(days=3)

    current = start
    days_checked = 0

    while current <= end:
        yyyy = current.year
        mm = current.month
        dd = current.day

        sitemap_url = f'https://www.aljazeera.com/sitemap.xml?yyyy={yyyy}&mm={mm:02d}&dd={dd:02d}'

        try:
            resp = requests.get(sitemap_url, headers=HEADERS, timeout=10)
            if resp.status_code == 200 and len(resp.content) > 100:
                root = ET.fromstring(resp.content)
                ns = {'ns': 'http://www.sitemaps.org/schemas/sitemap/0.9'}

                urls = [loc.text for loc in root.findall('.//ns:loc', ns) if loc.text]

                syria_kw = ['syria', 'syrian', 'assad', 'damascus', 'aleppo',
                            'idlib', 'hts', 'nusra', 'isis', 'isil', 'daesh',
                            'rebel', 'kurdi', 'turkey-syria', 'iran-syria',
                            'russia-syria', 'refugee']
                syria_urls = [
                    u for u in urls
                    if any(kw in u.lower() for kw in syria_kw)
                ]

                all_urls.extend(syria_urls)

                if syria_urls:
                    print(f'  {yyyy}-{mm:02d}-{dd:02d}: +{len(syria_urls)} (total: {len(all_urls)})')

            days_checked += 1
            if days_checked % 50 == 0:
                print(f'  ... checked {days_checked} days, {len(all_urls)} URLs found')

            time.sleep(0.3)

        except:
            pass

        current += delta

    return list(set(all_urls))

print(f'Fetching Al Jazeera sitemaps for {PHASE} ({START_DATE} to {END_DATE})...')
article_urls = get_aljazeera_from_sitemaps()
print(f'\n✅ Found {len(article_urls)} unique Syria-related URLs')

Fetching Al Jazeera sitemaps for P4 (2022-01-01 to 2025-12-31)...
  2022-01-07: +4 (total: 4)
  2022-01-10: +2 (total: 6)
  2022-01-13: +7 (total: 13)
  2022-01-16: +1 (total: 14)
  2022-01-19: +9 (total: 23)
  2022-01-22: +3 (total: 26)
  2022-01-25: +4 (total: 30)
  2022-01-28: +6 (total: 36)
  2022-01-31: +10 (total: 46)
  2022-02-03: +12 (total: 58)
  2022-02-06: +4 (total: 62)
  2022-02-09: +3 (total: 65)
  2022-02-12: +1 (total: 66)
  2022-02-15: +4 (total: 70)
  2022-02-18: +5 (total: 75)
  2022-02-21: +5 (total: 80)
  2022-02-24: +8 (total: 88)
  2022-02-27: +2 (total: 90)
  2022-03-02: +5 (total: 95)
  2022-03-05: +5 (total: 100)
  2022-03-08: +7 (total: 107)
  2022-03-11: +6 (total: 113)
  2022-03-14: +4 (total: 117)
  2022-03-17: +1 (total: 118)
  2022-03-23: +1 (total: 119)
  2022-03-26: +2 (total: 121)
  2022-03-29: +7 (total: 128)
  2022-04-01: +7 (total: 135)
  2022-04-04: +2 (total: 137)
  2022-04-07: +8 (total: 145)
  2022-04-10: +2 (total: 147)
  2022-04-13: +3 (total

In [ ]:
def scrape_article(url):
    """
    Extract title, date, and body text from an Al Jazeera article.
    """
    try:
        resp = requests.get(url, headers=HEADERS, timeout=15)
        if resp.status_code != 200:
            return None

        soup = BeautifulSoup(resp.text, 'html.parser')

        # Title
        title_tag = soup.find('h1')
        title = title_tag.get_text(strip=True) if title_tag else ''

        # Date — Al Jazeera uses <time> or meta tags
        date_str = ''
        time_tag = soup.find('time')
        if time_tag and time_tag.get('datetime'):
            date_str = time_tag['datetime'][:10]
        else:
            meta_date = soup.find('meta', {'property': 'article:published_time'})
            if meta_date:
                date_str = meta_date['content'][:10]

        # Body text
        # Al Jazeera wraps article text in <div class="wysiwyg"> or <main>
        body_div = soup.select_one('.wysiwyg--all-content, .article__body-text, main article')
        if body_div:
            paragraphs = body_div.find_all('p')
        else:
            paragraphs = soup.find_all('p')

        text = ' '.join(p.get_text(strip=True) for p in paragraphs if len(p.get_text(strip=True)) > 30)

        if not text or not title:
            return None

        return {
            'title': title,
            'date': date_str,
            'text': text,
            'url': url,
            'source': 'aljazeera_english'
        }

    except Exception as e:
        return None


# Scrape all articles
print(f'Scraping {len(article_urls)} articles...')
articles = []
errors = 0

for i, url in enumerate(article_urls):
    article = scrape_article(url)
    if article:
        articles.append(article)
    else:
        errors += 1

    if (i + 1) % 25 == 0:
        print(f'  {i+1}/{len(article_urls)} done — {len(articles)} scraped, {errors} failed')

    time.sleep(1)  # be respectful to the server

print(f'\n✅ Scraped {len(articles)} articles ({errors} failed)')

Scraping 1326 articles...
  25/1326 done — 18 scraped, 7 failed
  50/1326 done — 40 scraped, 10 failed
  75/1326 done — 61 scraped, 14 failed
  100/1326 done — 85 scraped, 15 failed
  125/1326 done — 108 scraped, 17 failed
  150/1326 done — 131 scraped, 19 failed
  175/1326 done — 154 scraped, 21 failed
  200/1326 done — 178 scraped, 22 failed
  225/1326 done — 199 scraped, 26 failed
  250/1326 done — 223 scraped, 27 failed
  275/1326 done — 247 scraped, 28 failed
  300/1326 done — 268 scraped, 32 failed
  325/1326 done — 291 scraped, 34 failed
  350/1326 done — 314 scraped, 36 failed
  375/1326 done — 338 scraped, 37 failed
  400/1326 done — 363 scraped, 37 failed
  425/1326 done — 385 scraped, 40 failed
  450/1326 done — 408 scraped, 42 failed
  475/1326 done — 432 scraped, 43 failed
  500/1326 done — 456 scraped, 44 failed
  525/1326 done — 478 scraped, 47 failed
  550/1326 done — 501 scraped, 49 failed
  575/1326 done — 523 scraped, 52 failed
  600/1326 done — 545 scraped, 55 faile

In [ ]:
df_aj = pd.DataFrame(articles)

print(f'Columns found: {df_aj.columns.tolist()}')
print(f'Total rows: {len(df_aj)}')

# if 'date' not in df_aj.columns or df_aj['date'].isna().all() or (df_aj['date'] == '').all():
#     print('⚠️  Extracting dates from URLs...')
#     def extract_date_from_url(url):
#         match = re.search(r'/(\d{4})/(\d{1,2})/(\d{1,2})/', str(url))
#         if match:
#             return f'{match.group(1)}-{int(match.group(2)):02d}-{int(match.group(3)):02d}'
#         return None
#     df_aj['date'] = df_aj['url'].apply(extract_date_from_url)
df_aj = pd.DataFrame(articles)
print(f'Columns found: {df_aj.columns.tolist()}')
print(f'Total rows: {len(df_aj)}')

# Always try to fill missing dates from URLs
def extract_date_from_url(url):
    match = re.search(r'/(\d{4})/(\d{1,2})/(\d{1,2})/', str(url))
    if match:
        return f'{match.group(1)}-{int(match.group(2)):02d}-{int(match.group(3)):02d}'
    return None

# Fill in dates: use existing date if present, otherwise extract from URL
if 'date' not in df_aj.columns:
    df_aj['date'] = None

url_dates = df_aj['url'].apply(extract_date_from_url)
df_aj['date'] = df_aj['date'].fillna(url_dates)
df_aj['date'] = df_aj['date'].replace('', None).fillna(url_dates)

print(f'   Dates from scraper: {df_aj["date"].notna().sum()}')
print(f'   Missing dates: {df_aj["date"].isna().sum()}')
df_aj['date'] = pd.to_datetime(df_aj['date'], errors='coerce')
df_aj = df_aj.dropna(subset=['date'])
df_aj = df_aj[(df_aj['date'] >= START_DATE) & (df_aj['date'] <= END_DATE)]
df_aj['year'] = df_aj['date'].dt.year
df_aj['date'] = df_aj['date'].dt.strftime('%Y-%m-%d')

syria_mask = df_aj['text'].str.contains('syria|syrian|assad|damascus', case=False, na=False)
df_aj = df_aj[syria_mask]

df_aj['section'] = 'news'
df_aj['phase'] = PHASE
df_aj['source'] = 'aljazeera_english'
df_aj['id'] = [f'aj_{PHASE}_{i}' for i in range(len(df_aj))]

print(f'\n✅ {len(df_aj)} Al Jazeera articles for {PHASE}')
if len(df_aj) > 0:
    print(f'   Date range: {df_aj["date"].min()} to {df_aj["date"].max()}')
    print(df_aj['year'].value_counts().sort_index().to_string())

# Save with phase in filename
out_path = os.path.join(OUTPUT_FOLDER, f'aljazeera_syria_{PHASE.lower()}.csv')
df_aj.to_csv(out_path, index=False)
print(f'\n💾 Saved: {out_path}')

Columns found: ['title', 'date', 'text', 'url', 'source']
Total rows: 1204
Columns found: ['title', 'date', 'text', 'url', 'source']
Total rows: 1204
   Dates from scraper: 1204
   Missing dates: 0

✅ 480 Al Jazeera articles for P4
   Date range: 2022-01-07 to 2025-12-29
year
2022    114
2023    144
2024    101
2025    121

💾 Saved: /content/drive/MyDrive/arc_of_conflict/data/aljazeera_syria_p4.csv
